# Module A — Dataset Construction & Indexing

1. 2500 data per language
2. 5 English and 5 Bangla newspaper
3. Crawling done using BeautifulSoup and RSS
4. Metadata : a. title
              b. body
              c. date
              d. url
              e. language
              f. token number

In [3]:
import feedparser
import json
from tqdm import tqdm
import os
from bs4 import BeautifulSoup
import html
import re

## HTML to Text

In [4]:
def html_to_text(s):
    if not s:
        return ""
    s = html.unescape(s)
    soup = BeautifulSoup(s, "html.parser")
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 1. Data Crawling 

### 1.1 Collect RSS feeds

In [5]:
def collect_from_rss_feeds(rss_feeds, max_docs=200, doc_prefix="en"):
    docs = []
    seen_urls = set()
    doc_i = 0

    for rss_url in tqdm(rss_feeds, desc="Processing RSS feeds"):
        feed = feedparser.parse(rss_url)

        for entry in feed.entries:
            url = getattr(entry, "link", "").strip()
            if not url or url in seen_urls:
                continue

            title = getattr(entry, "title", "").strip()
            summary_raw = getattr(entry, "summary", "").strip()
            summary_text = html_to_text(summary_raw)

            date = getattr(entry, "published", "")

            seen_urls.add(url)

            docs.append({
                "doc_id": f"{doc_prefix}_{doc_i:06d}",
                "title": html_to_text(title),
                "body": summary_text,
                "url": url,
                "date": date,
                "language": doc_prefix,
                "token_count": len(summary_text.split())
            })

            doc_i += 1

            if len(docs) >= max_docs:
                return docs

    return docs

### 1.2 English Newspapers URLs

In [6]:
rss_feeds=[
    "https://www.thedailystar.net/rss.xml",
    "https://www.dhakatribune.com/feed/",
    "https://dailynewnation.com/feed/",
    "https://thebangladeshtoday.com/?feed=rss2",
    "https://dailyasianage.com/rss/feed.xml",
    "https://www.thedailystar.net/historical/front-page/rss.xml",
    "https://www.thedailystar.net/business/rss.xml",
    "https://www.thedailystar.net/science-tech/rss.xml",
    "https://www.thedailystar.net/sports/rss.xml",
    "https://www.thedailystar.net/opinion/rss.xml",
    "https://www.thedailystar.net/world/rss.xml",
    "https://www.thedailystar.net/country/rss.xml",
    "https://www.thedailystar.net/environment/rss.xml",
    "https://www.thedailystar.net/arts-culture/rss.xml",
    "https://www.thedailystar.net/magazine/rss.xml",
    "https://www.thedailystar.net/backpage/rss.xml",
    "https://www.thedailystar.net/star-weekend/rss.xml",
    "https://www.thedailystar.net/star-multimedia/rss.xml",
]

### 1.3 Data Collecting from English Newspapers in JSON file

In [7]:
saving_path = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data"
file_path = os.path.join(saving_path, "document_en.json")

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        existing_docs = json.load(f)
else:
    existing_docs = []

existing_urls = {doc["url"] for doc in existing_docs}

new_docs = collect_from_rss_feeds(rss_feeds)
new_docs = [doc for doc in new_docs if doc["url"] not in existing_urls]

start_id = len(existing_docs)
for i, doc in enumerate(new_docs):
    doc["doc_id"] = f"en_{start_id + i:06d}"

all_docs = existing_docs + new_docs

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

print(f"Added {len(new_docs)} new documents. Total: {len(all_docs)}")

Processing RSS feeds:  50%|█████     | 9/18 [00:11<00:11,  1.31s/it]

Added 200 new documents. Total: 200


### 1.4 Bangla Newspapers URLs

In [8]:
rss_feeds=[
    "https://www.risingbd.com/rss/rss.xml",
    "https://bd-journal.com/feed/latest-rss.xml",
    "https://bangladeshdiplomat.com/feed",
    "https://www.jagonews24.com/rss/rss.xml",
    "https://bdpratidin.net/rss/latest-posts",
    "https://www.kalerkantho.com/rss.xml",
    "https://www.banglatribune.com/feed/",
    "https://bangla.thedailystar.net/rss.xml"
]

### 1.5 Data Collecting from Bangla Newspapers in JSON file

In [9]:
saving_path = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data"
file_path = os.path.join(saving_path, "document_bn.json")

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        existing_docs = json.load(f)
else:
    existing_docs = []

existing_urls = {doc["url"] for doc in existing_docs}

new_docs = collect_from_rss_feeds(rss_feeds)
new_docs = [doc for doc in new_docs if doc["url"] not in existing_urls]

start_id = len(existing_docs)
for i, doc in enumerate(new_docs):
    doc["doc_id"] = f"bn_{start_id + i:06d}"

all_docs = existing_docs + new_docs

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)

print(f"Added {len(new_docs)} new documents. Total: {len(all_docs)}")

Processing RSS feeds:  50%|█████     | 4/8 [00:08<00:08,  2.09s/it]

Added 200 new documents. Total: 200


Purpose

1. Gain exposure to real-world, messy data
2. Understand indexing fundamentals
3. Create a foundation for multilingual search